In [1]:
import pandas as pd
import plotly.express as px
from patent_retrieval import utils as utils,dataset as dataset
from pathlib import Path
import os
import json
import sqlmodel as sqlm

In [2]:
test_topics_path=(Path(os.environ["CLEF_IP_LOCATION"])/ "02_topics"/ "test-pac"/ "relass_clef-ip-2011-PAC_abs.txt")

results_df = pd.read_csv('data/patent_retrieval_final.csv')
results_df

,Name,State,Notes,User,Tags,Created,Runtime,Sweep,backend,doc_columns,...,precision@50,precision@500,recall@10,recall@100,recall@1000,recall@20,recall@200,recall@300,recall@50,recall@500
0,qwen3-emb-8b_db-v4_all-topics_abstract_aysm_to...,finished,-,NaN,instruct,2026-02-13T19:33:46.000Z,7069,NaN,openai,"[""title"",""abstract""]",...,0.031231,0.006257,0.168375,0.425890,0.735470,0.228096,0.523879,0.583461,0.335080,0.650012
1,bm25_db-v4_all-topics_abstract-claims_aysm_top...,finished,-,NaN,instruct,2026-02-13T19:09:48.000Z,4131,NaN,NaN,"[""title"",""abstract"",""claims""]",...,0.016192,0.002731,0.099641,0.194462,0.306999,0.124739,0.226732,0.245524,0.162666,0.271338
2,bm25_db-v4_all-topics_abstract_aysm_top1000,finished,-,NaN,instruct,2026-02-13T19:08:21.000Z,1922,NaN,NaN,"[""title"",""abstract""]",...,0.011906,0.002119,0.072903,0.146471,0.241071,0.092576,0.173390,0.189381,0.121730,0.209791
3,bge-m3_db-v4_all-topics_abstract_aysm_top1000,finished,-,NaN,instruct,2026-02-13T14:57:26.000Z,6173,NaN,openai,"[""title"",""abstract""]",...,0.015235,0.003104,0.084639,0.198988,0.380227,0.112570,0.247009,0.278786,0.157258,0.316539
4,bge-m3_db-v4_all-topics_abstract-claims_aysm_t...,finished,-,NaN,instruct,2026-02-13T14:57:22.000Z,3764,NaN,openai,"[""title"",""abstract"",""claims""]",...,0.018237,0.003544,0.105712,0.234604,0.433653,0.137640,0.285459,0.320487,0.190479,0.364691
5,qwen3-emb-8b_db-v4_all-topics_abstract-claims_...,finished,-,NaN,instruct,2026-02-13T11:51:06.000Z,12627,NaN,openai,"[""title"",""abstract"",""claims""]",...,0.037633,0.007045,0.213638,0.509711,0.810072,0.287259,0.610558,0.667375,0.403876,0.732470
6,qwen3-emb-4b_db-v4_all-topics_abstract-claims_...,finished,-,NaN,instruct,2026-02-13T11:50:35.000Z,5100,NaN,openai,"[""title"",""abstract"",""claims""]",...,0.034052,0.006458,0.189458,0.445348,0.755473,0.254813,0.540621,0.595884,0.358962,0.668356
7,llama-nemo-8b_db-v4_all-topics_abstract-claims...,finished,-,NaN,instruct,2026-02-12T23:53:30.000Z,42650,NaN,openai,"[""title"",""abstract"",""claims""]",...,0.036867,0.006914,0.209754,0.487923,0.800625,0.280821,0.587652,0.645030,0.392311,0.713423
8,llama-nemo-8b_db-v4_all-topics_abstract_aysm_t...,finished,-,NaN,instruct,2026-02-12T23:52:04.000Z,37391,NaN,openai,"[""title"",""abstract""]",...,0.031755,0.006409,0.174040,0.432785,0.754844,0.237348,0.534133,0.593871,0.339691,0.663669
9,patQwen3-emb-4b-v2_db-v4_all-topics_abstract_a...,finished,-,NaN,instruct,2026-02-12T22:56:25.000Z,14778,NaN,openai,"[""title"",""abstract""]",...,0.033478,0.006415,0.185666,0.452003,0.754068,0.252207,0.550692,0.604009,0.361114,0.671068


In [3]:
results_df = results_df.iloc[10:11]
results_df

,Name,State,Notes,User,Tags,Created,Runtime,Sweep,backend,doc_columns,...,precision@50,precision@500,recall@10,recall@100,recall@1000,recall@20,recall@200,recall@300,recall@50,recall@500
10,patQwen3-emb-4b-v2_db-v4_all-topics_abstract-c...,finished,-,NaN,instruct,2026-02-12T22:55:47.000Z,11436,NaN,openai,"[""title"",""abstract"",""claims""]",...,0.041999,0.007298,0.242167,0.554667,0.835267,0.325191,0.648589,0.700074,0.451966,0.762802


In [4]:
db_path=os.path.join(os.environ["CLEF_IP_LOCATION"], "patents_v4.db")
engine = sqlm.create_engine(f"sqlite:///{db_path}")


In [5]:
test_df = pd.read_csv(test_topics_path, sep="\t", header=None, names=["topic_number", "candidate_number", "score"])
test_df


,topic_number,candidate_number,score
0,EP-1221372-A2,EP-0229673,1
1,EP-1221372-A2,EP-0271257,1
2,EP-1221372-A2,EP-0445916,1
3,EP-1221372-A2,EP-0512799,1
4,EP-1221372-A2,EP-0547921,1
...,...,...,...
19557,EP-1936237-A1,EP-0252423,2
19558,EP-1936237-A1,EP-0320621,1
19559,EP-1936750-A1,EP-0948090,2
19560,EP-1936750-A1,EP-0948092,2


In [6]:
details_results = results_df[['embedding_model', 'output_dir']]
#true_df = pd.read_csv(    test_topics_path,    names=["topic_number", "candidate_number", "score"],    sep="\t",    )
true_dict = test_df.groupby("topic_number")["candidate_number"].apply(set).to_dict()

all_details = None

for idx, row in details_results.iterrows():
    print(f"Details for {row['embedding_model']} in {row['output_dir']}:")
    detail_df = pd.read_csv(f"{row['output_dir']}/results.csv")#.groupby(['topic','number']).head(200).reset_index()
    detail_df['embedding_model'] = row['embedding_model']
    detail_df = detail_df.groupby(["embedding_model","topic"]).apply(lambda x: x.sort_values('score', ascending=False).head(100)).reset_index(drop=True)
    # Compute recall per topic using lambda
    detail_df['is_relevant'] = detail_df.apply(
    lambda x: x['number'] in true_dict.get(x['topic'], set()), axis=1
    )
    recall_per_topic = detail_df.groupby(["embedding_model","topic"]).apply(
        lambda group: group['is_relevant'].sum() / len(true_dict.get(group.name[1], set())) if len(true_dict.get(group.name[1], set())) > 0 else 0
    ).reset_index(name='recall')

    if all_details is None:
        all_details = recall_per_topic
    else:
        all_details = pd.concat([all_details, recall_per_topic], ignore_index=True)    


Details for patQwen3-emb-4b in /home/alm3rng/patent-retrieval/embeddings/runs/patQwen3-emb-4b-v2_db-v4_all-topics_abstract-claims_aysm_top1000:


/tmp/ipykernel_1440081/1464351235.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  detail_df = detail_df.groupby(["embedding_model","topic"]).apply(lambda x: x.sort_values('score', ascending=False).head(100)).reset_index(drop=True)
/tmp/ipykernel_1440081/1464351235.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  recall_per_topic = detail_df.groupby(["embedding_model","topic"]).apply(


In [7]:
recall0_topics = all_details[(all_details["recall"]==0)]#& (all_details["embedding_model"]=="patQwen3-emb-4b")].reset_index(drop=True)
recall0_topics

,embedding_model,topic,recall
14,patQwen3-emb-4b,EP-1225535-A2,0.0
17,patQwen3-emb-4b,EP-1225761-A1,0.0
61,patQwen3-emb-4b,EP-1233101-A2,0.0
64,patQwen3-emb-4b,EP-1233359-A2,0.0
90,patQwen3-emb-4b,EP-1235465-A2,0.0
...,...,...,...
3925,patQwen3-emb-4b,EP-1923119-A1,0.0
3927,patQwen3-emb-4b,EP-1923208-A1,0.0
3943,patQwen3-emb-4b,EP-1929944-A2,0.0
3952,patQwen3-emb-4b,EP-1931179-A1,0.0


In [ ]:
recall0_topics

In [9]:
document_collection_dir=Path(os.environ["CLEF_IP_LOCATION"]) / "01_document_collection" / "document_collection_pac"

def parse_candidate_path(patent_id: str): 

    if patent_id[:2] == 'EP':
        return f"{patent_id[:2]}/{"000000" if patent_id[3] == '0' else "000001"}/{patent_id[4:6]}/{patent_id[6:8]}/{patent_id[8:10]}/{patent_id}*.xml"
    else:
        return f"{patent_id[:2]}/00{patent_id[3:7]}/{patent_id[7:9]}/{patent_id[9:11]}/{patent_id[11:13]}/{patent_id}*.xml"


def find_patent_file(identifier: str) :
    # first try the test topics PAC_topics/files (same approach as get_patent)

    topic_glob = list(test_topics_path.parent.glob(f"PAC_topics/files/{identifier}*.xml"))

    if topic_glob:
        return topic_glob[0]
    # then try the document collection dirs (document_collection_dir is a tuple)

    topic_glob = list(document_collection_dir.glob(parse_candidate_path(identifier)))
    #print(parse_candidate_path(identifier))
    #print(topic_glob)
    #print(topic_glob)
    return topic_glob[0]


In [ ]:
recall0_topics['topic_language'] = recall0_topics['topic'].apply(lambda x: dataset.parse_patent([find_patent_file(x)])[0].language)
recall0_topics

/tmp/ipykernel_1440081/1930576060.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recall0_topics['language'] = recall0_topics['topic'].apply(lambda x: dataset.parse_patent([find_patent_file(x)])[0].language)


,embedding_model,topic,recall,language
14,patQwen3-emb-4b,EP-1225535-A2,0.0,DE
17,patQwen3-emb-4b,EP-1225761-A1,0.0,FR
61,patQwen3-emb-4b,EP-1233101-A2,0.0,DE
64,patQwen3-emb-4b,EP-1233359-A2,0.0,DE
90,patQwen3-emb-4b,EP-1235465-A2,0.0,DE
...,...,...,...,...
3925,patQwen3-emb-4b,EP-1923119-A1,0.0,FR
3927,patQwen3-emb-4b,EP-1923208-A1,0.0,FR
3943,patQwen3-emb-4b,EP-1929944-A2,0.0,DE
3952,patQwen3-emb-4b,EP-1931179-A1,0.0,FR


In [18]:
recall0_topics

,embedding_model,topic,recall,topic_language
14,patQwen3-emb-4b,EP-1225535-A2,0.0,DE
17,patQwen3-emb-4b,EP-1225761-A1,0.0,FR
61,patQwen3-emb-4b,EP-1233101-A2,0.0,DE
64,patQwen3-emb-4b,EP-1233359-A2,0.0,DE
90,patQwen3-emb-4b,EP-1235465-A2,0.0,DE
...,...,...,...,...
3925,patQwen3-emb-4b,EP-1923119-A1,0.0,FR
3927,patQwen3-emb-4b,EP-1923208-A1,0.0,FR
3943,patQwen3-emb-4b,EP-1929944-A2,0.0,DE
3952,patQwen3-emb-4b,EP-1931179-A1,0.0,FR


In [22]:

# Sample 5 recall=0 topics and inspect query text
sample_topics = recall0_topics["topic"].sample(min(5, len(recall0_topics)), random_state=42).tolist()

for topic_id in sample_topics:
    topic_file = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    topic_patent = dataset.parse_patent([topic_file])[0]
    query = dataset.extract_query_text(topic_patent, ["abstract", "claims"])

    lang = (topic_patent.language or "NONE").lower()
    abs_native = getattr(topic_patent, f"abstract_{lang}", None) if lang != "none" else None
    abs_en = topic_patent.abstract_en
    claims_native = getattr(topic_patent, f"claims_{lang}", None) if lang != "none" else None
    claims_en = topic_patent.claims_en

    print(f"--- {topic_id} | lang={topic_patent.language} | #rel={len(true_dict.get(topic_id, set()))}")
    print(f"  abs_{lang}: {'EMPTY' if not abs_native else str(abs_native)[:80]}")
    print(f"  abs_en:  {'EMPTY' if not abs_en else str(abs_en)[:80]}")
    print(f"  clm_{lang}: {'EMPTY' if not claims_native else f'{len(claims_native)} claims'}")
    print(f"  clm_en:  {'EMPTY' if not claims_en else f'{len(claims_en)} claims'}")
    print(f"  query_len={len(query)}, preview: {query[:200]}")
    print()


/tmp/ipykernel_2345383/4178014840.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recall0_topics['language'] = recall0_topics['topic'].apply(lambda x: dataset.parse_patent([find_patent_file(x)])[0].language)


Language distribution of recall=0 topics:
language
FR    150
DE    140
EN    119
Name: count, dtype: int64

Total recall=0 topics: 409

English recall=0 topics: 119

--- EP-1251440-A2 | #relevant=4 | abs_len=784 | n_claims=137 | query_len=61108
  Abstract: A token type content providing system is provided, in which the system includes a portable user terminal and a linkup server, the portable user termin
  Claim 1:  A token type content providing system comprising a portable user terminal and a linkup server, said portable user terminal characterized by: a part fo
  IPC: ['G06F  17/30        20060101A I20051008RMEP', 'G06F  17/30        20060101C I20051008RMEP']

--- EP-1255207-A2 | #relevant=4 | abs_len=859 | n_claims=7 | query_len=4337
  Abstract: A method and apparatus for automatically accessing a WWW page is provided for realizing a function of automatically jumping to a desired page to autom
  Claim 1:  A method of automatically searching a hypertext structure for use with a web 

In [23]:

# Compare with 3 high-recall topics
high_recall = all_details[all_details["recall"] >= 0.5]["topic"].sample(min(3, len(all_details[all_details["recall"] >= 0.5])), random_state=42).tolist()

for topic_id in high_recall:
    try:
        topic_file = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    except StopIteration:
        continue
    topic_patent = dataset.parse_patent([topic_file])[0]
    query = dataset.extract_query_text(topic_patent, ["abstract", "claims"])

    print(f"=== {topic_id} | language={topic_patent.language} | recall={all_details[all_details['topic']==topic_id]['recall'].values[0]:.2f} ===")
    print(f"  QUERY LENGTH: {len(query)} chars")
    print(f"  QUERY PREVIEW:\n{query[:500]}")
    print()


=== EP-1251440-A2 | query_len=61108 | #relevant=4 ===
  Topic abstract: A token type content providing system is provided, in which the system includes a portable user term
  Cand WO-1998049813 lang=EN: abs_en=A destination website access system employs a bar code reader cooperating with a personal computer o | claims_en=12
  Cand WO-1998008056 lang=EN: abs_en=A navigation aid for use with printed maps, the aid comprising: (a) a receiver of transmitted signal | claims_en=5
  Cand EP-0987868 lang=EN: abs_en=The present invention relates to navigation of the Internet by a two-way
interactive communication m | claims_en=14
  Cand WO-1999066441 lang=EN: abs_en=An interactive data transfer system and method is provided. In embodiments of the invention, the dat | claims_en=22

=== EP-1255207-A2 | query_len=4337 | #relevant=4 ===
  Topic abstract: A method and apparatus for automatically accessing a WWW page is provided for realizing a function o
  Cand WO-2001009766 lang=EN: abs_en=Disclosed

In [ ]:

# Key observation: queries for recall=0 topics are in FR but have English abstracts available too.
# The retriever uses the native-language text. Check: are documents indexed in English only?

# Verify: for failing FR topics, the query is in French, but candidates have English text in DB
# Check what language the *candidates* have for one failing topic
sample_fail = "EP-1418478-A1"
cand_ids = list(true_dict.get(sample_fail, set()))[:]
print(f"Candidates for failing topic {sample_fail}: {cand_ids}")

with sqlm.Session(engine) as session:
    for cid in cand_ids:
        stmt = sqlm.select(dataset.Patent.number, dataset.Patent.language, 
                           dataset.Patent.abstract_en, dataset.Patent.abstract_fr).where(
            dataset.Patent.number == cid)
        r = session.exec(stmt).first()
        if r:
            print(f"  {r[0]} lang={r[1]}, abs_en={'YES' if r[2] else 'NO'}, abs_fr={'YES' if r[3] else 'NO'}")
        else:
            print(f"  {cid} NOT IN DB")


In [13]:
recall0_topics

,embedding_model,topic_language,recall,language
14,patQwen3-emb-4b,EP-1225535-A2,0.0,DE
17,patQwen3-emb-4b,EP-1225761-A1,0.0,FR
61,patQwen3-emb-4b,EP-1233101-A2,0.0,DE
64,patQwen3-emb-4b,EP-1233359-A2,0.0,DE
90,patQwen3-emb-4b,EP-1235465-A2,0.0,DE
...,...,...,...,...
3925,patQwen3-emb-4b,EP-1923119-A1,0.0,FR
3927,patQwen3-emb-4b,EP-1923208-A1,0.0,FR
3943,patQwen3-emb-4b,EP-1929944-A2,0.0,DE
3952,patQwen3-emb-4b,EP-1931179-A1,0.0,FR


In [19]:
from collections import Counter

Counter(recall0_topics['topic_language'])

Counter({'FR': 150, 'DE': 140, 'EN': 119})

In [16]:
with sqlm.Session(engine) as session:
    query = sqlm.select(
        dataset.Patent.number
    )
    
    # Execute using session.exec()
    results = session.exec(query).all()

In [21]:

test0_df = test_df[test_df["topic_number"].isin(recall0_topics["topic"])]
candidate_numbers = test0_df["candidate_number"].dropna().unique().tolist()

rows = []
chunk_size = 900  # keep below SQLite variable limit

with sqlm.Session(engine) as session:
    for i in range(0, len(candidate_numbers), chunk_size):
        chunk = candidate_numbers[i:i + chunk_size]
        stmt = (
            sqlm.select(dataset.Patent.number, dataset.Patent.language)
            .where(dataset.Patent.number.in_(chunk))
        )
        rows.extend(session.exec(stmt).all())

candidate_language_df = (
    pd.DataFrame(rows, columns=["candidate_number", "cand_language"])
    .drop_duplicates(subset=["candidate_number"])
)

test_df_with_language = test0_df.merge(candidate_language_df, on="candidate_number", how="left")
test_df_with_language["topic_language"] = test_df_with_language["topic_number"].map(recall0_topics.set_index("topic")["topic_language"])
test_df_with_language

,topic_number,candidate_number,score,cand_language,topic_language
0,EP-1225535-A2,EP-0238043,1,DE,DE
1,EP-1225535-A2,EP-0490412,1,DE,DE
2,EP-1225535-A2,EP-1014201,2,DE,DE
3,EP-1225761-A1,EP-0837599,1,EN,FR
4,EP-1225761-A1,EP-0996307,2,EN,FR
...,...,...,...,...,...
1533,EP-1935403-A2,EP-1175893,1,EN,FR
1534,EP-1935403-A2,WO-1993017060,1,EN,FR
1535,EP-1935403-A2,WO-1996012754,1,EN,FR
1536,EP-1935403-A2,WO-2001014458,2,EN,FR


In [11]:
candidate_ids = test_df_with_language["candidate_number"].dropna().unique().tolist()

# Resolve column names in dataset.Patent defensively
patent_cols = set(dataset.Patent.__table__.columns.keys())
abstract_col = next((c for c in ["abstract_en", "abstract_de", "abstract_fr"] if c in patent_cols), None)
claims_col = next((c for c in ["claims_en", "claims_de", "claims_fr"] if c in patent_cols), None)

if abstract_col is None or claims_col is None:
    raise ValueError(f"Could not find abstract/claims columns. Available columns: {sorted(patent_cols)}")

rows = []
chunk_size_db = 900

with sqlm.Session(engine) as session:
    for start in range(0, len(candidate_ids), chunk_size_db):
        chunk_ids = candidate_ids[start:start + chunk_size_db]
        stmt = (
            sqlm.select(
                dataset.Patent.number,
                getattr(dataset.Patent, abstract_col),
                getattr(dataset.Patent, claims_col),
            )
            .where(dataset.Patent.number.in_(chunk_ids))
        )
        rows.extend(session.exec(stmt).all())

candidate_content_df = pd.DataFrame(
    rows, columns=["candidate_number", "abstract", "claims"]
).drop_duplicates("candidate_number")

def _is_missing_text(x):
    if x is None:
        return True
    if isinstance(x, str):
        s = x.strip().lower()
        return s == "" or s in {"nan", "none", "null", "[]", "{}"}
    if isinstance(x, (list, tuple, set)):
        return len(x) == 0 or all(_is_missing_text(v) for v in x)
    if isinstance(x, dict):
        return len(x) == 0
    try:
        return bool(pd.isna(x))
    except Exception:
        return False

candidate_content_df["missing_abstract"] = candidate_content_df["abstract"].apply(_is_missing_text)
candidate_content_df["missing_claims"] = candidate_content_df["claims"].apply(_is_missing_text)
candidate_content_df["missing_both"] = (
    candidate_content_df["missing_abstract"] & candidate_content_df["missing_claims"]
)

candidate_missing_df = (
    pd.DataFrame({"candidate_number": candidate_ids})
    .merge(
        candidate_content_df[["candidate_number", "missing_abstract", "missing_claims", "missing_both"]],
        on="candidate_number",
        how="left",
    )
)

candidate_missing_df["not_in_db"] = candidate_missing_df["missing_abstract"].isna()
candidate_missing_df.loc[
    candidate_missing_df["not_in_db"], ["missing_abstract", "missing_claims", "missing_both"]
] = True

total_candidates = len(candidate_missing_df)
missing_summary = pd.DataFrame(
    {
        "metric": ["total_candidates", "missing_abstract", "missing_claims", "missing_both", "not_in_db"],
        "count": [
            total_candidates,
            int(candidate_missing_df["missing_abstract"].sum()),
            int(candidate_missing_df["missing_claims"].sum()),
            int(candidate_missing_df["missing_both"].sum()),
            int(candidate_missing_df["not_in_db"].sum()),
        ],
    }
)
missing_summary["pct"] = (
    (missing_summary["count"] / total_candidates * 100).round(2) if total_candidates else 0.0
)

test_df_with_language_missing = test_df_with_language.merge(
    candidate_missing_df, on="candidate_number", how="left"
)

missing_summary

,metric,count,pct
0,total_candidates,1509,100.00
1,missing_abstract,51,3.38
2,missing_claims,255,16.90
3,missing_both,14,0.93
4,not_in_db,0,0.00


In [12]:
candidate_missing_df

,candidate_number,missing_abstract,missing_claims,missing_both,not_in_db
0,EP-0238043,False,False,False,False
1,EP-0490412,False,False,False,False
2,EP-1014201,False,True,False,False
3,EP-0837599,False,False,False,False
4,EP-0996307,False,False,False,False
...,...,...,...,...,...
1504,EP-1175893,False,False,False,False
1505,WO-1993017060,False,False,False,False
1506,WO-1996012754,False,False,False,False
1507,WO-2001014458,False,False,False,False


In [ ]:
rows_all[0]

1174264

In [47]:
cross_lingual = test_df_with_language.copy()
cross_lingual['is_cross_lingual'] = cross_lingual['topic_language'] != cross_lingual['cand_language']

cross_lingual_summary = cross_lingual.groupby(['topic_language', 'cand_language']).size().reset_index(name='count')

# Add percentage per topic_language group
cross_lingual_summary['pct'] = cross_lingual_summary.groupby('topic_language')['count'].transform(lambda x: x / x.sum() * 100)
cross_lingual_summary['pct_label'] = cross_lingual_summary['pct'].map(lambda x: f"{x:.1f}%")

fig = px.bar(
    cross_lingual_summary,
    x='topic_language',
    y='count',
    color='cand_language',
    barmode='group',
    text='pct_label',
    title='Cross-linguality between Topic and Candidate Languages',
    labels={'topic_language': 'Topic Language', 'cand_language': 'Candidate Language', 'count': 'Count'},
    height=600
)
fig.update_traces(textposition='outside')
fig.show()

In [52]:
recall_not0_topics = all_details[(all_details["recall"]>=0.5)]#& (all_details["embedding_model"]=="patQwen3-emb-4b")]
recall_not0_topics

,embedding_model,topic,recall
1,patQwen3-emb-4b,EP-1222860-A2,0.714286
3,patQwen3-emb-4b,EP-1223003-A1,0.571429
4,patQwen3-emb-4b,EP-1223076-A1,1.000000
5,patQwen3-emb-4b,EP-1223118-A1,0.750000
8,patQwen3-emb-4b,EP-1225199-A1,0.666667
...,...,...,...
3958,patQwen3-emb-4b,EP-1932613-A1,1.000000
3959,patQwen3-emb-4b,EP-1933055-A2,1.000000
3960,patQwen3-emb-4b,EP-1933117-A2,0.750000
3967,patQwen3-emb-4b,EP-1935732-A1,1.000000


In [53]:
recall_not0_topics.describe()

,recall
count,2429.000000
mean,0.766552
std,0.187343
min,0.500000
25%,0.625000
50%,0.750000
75%,1.000000
max,1.000000


In [51]:
all_details.describe()

,recall
count,3971.000000
mean,0.554667
std,0.317749
min,0.000000
25%,0.333333
50%,0.571429
75%,0.800000
max,1.000000


In [25]:

# Compare query characteristics: recall=0 vs recall>0 (sampled)
import numpy as np

def get_query_stats(topic_id):
    try:
        tf = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    except StopIteration:
        return None
    tp = dataset.parse_patent([tf])[0]
    q = dataset.extract_query_text(tp, ["abstract", "claims"])
    lang = (tp.language or 'en').lower()
    abs_text = getattr(tp, f"abstract_{lang}", None) or ''
    claims_list = getattr(tp, f"claims_{lang}", None) or []
    return {
        'topic': topic_id,
        'language': tp.language,
        'query_len': len(q),
        'abs_len': len(abs_text),
        'n_claims': len(claims_list) if isinstance(claims_list, list) else 0,
        'n_relevant': len(true_dict.get(topic_id, set())),
    }

# Sample from recall=0 and recall>0 buckets
r0 = recall0_topics['topic'].tolist()
r_pos = all_details[all_details['recall'] > 0]['topic'].tolist()

sample_r0 = np.random.RandomState(42).choice(r0, min(50, len(r0)), replace=False)
sample_rpos = np.random.RandomState(42).choice(r_pos, min(50, len(r_pos)), replace=False)

stats_r0 = pd.DataFrame([s for t in sample_r0 if (s := get_query_stats(t)) is not None])
stats_r0['group'] = 'recall=0'
stats_rpos = pd.DataFrame([s for t in sample_rpos if (s := get_query_stats(t)) is not None])
stats_rpos['group'] = 'recall>0'

stats_df = pd.concat([stats_r0, stats_rpos], ignore_index=True)

print("=== Median query stats ===")
print(stats_df.groupby('group')[['query_len', 'abs_len', 'n_claims', 'n_relevant']].median().round(0))
print("\n=== Mean query stats ===")
print(stats_df.groupby('group')[['query_len', 'abs_len', 'n_claims', 'n_relevant']].mean().round(0))
print("\n=== Language distribution ===")
print(stats_df.groupby(['group', 'language']).size().unstack(fill_value=0))


=== Median query stats ===
          query_len  abs_len  n_claims  n_relevant
group                                             
recall=0     4592.0    732.0      13.0         3.0
recall>0     4754.0    680.0      13.0         4.0

=== Mean query stats ===
          query_len  abs_len  n_claims  n_relevant
group                                             
recall=0     5668.0    745.0      15.0         3.0
recall>0     7453.0    707.0      20.0         6.0

=== Language distribution ===
language  DE  EN  FR
group               
recall=0  21  13  16
recall>0  13  22  15


In [27]:

# Check topic vs candidate abstracts for EN recall=0 topics
en_fail = recall0_topics[recall0_topics['language'] == 'EN']['topic'].tolist()[:3]

for topic_id in en_fail:
    tf = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    tp = dataset.parse_patent([tf])[0]
    cand_ids = list(true_dict.get(topic_id, set()))
    
    print(f"=== {topic_id} ({len(cand_ids)} rel) ===")
    print(f"  TOPIC: {(tp.abstract_en or '')[:150]}")
    
    with sqlm.Session(engine) as session:
        for cid in cand_ids[:3]:
            stmt = sqlm.select(dataset.Patent.abstract_en, dataset.Patent.language).where(dataset.Patent.number == cid)
            r = session.exec(stmt).first()
            if r and r[0]:
                print(f"  CAND {cid} ({r[1]}): {r[0][:150]}")
            else:
                print(f"  CAND {cid}: NO EN ABSTRACT")
    print()


=== EP-1251440-A2 (4 rel) ===
  TOPIC: A token type content providing system is provided, in which the system includes a portable user terminal and a linkup server, the portable user termin
  CAND WO-1998049813 (EN): A destination website access system employs a bar code reader cooperating with a personal computer or workstation for accessing a unique destination w
  CAND WO-1998008056 (EN): A navigation aid for use with printed maps, the aid comprising: (a) a receiver of transmitted signals from a geographic position fixing system such as
  CAND EP-0987868 (EN): The present invention relates to navigation of the Internet by a two-way
interactive communication mobile device capable of wireless communication, vi

=== EP-1255207-A2 (4 rel) ===
  TOPIC: A method and apparatus for automatically accessing a WWW page is provided for realizing a function of automatically jumping to a desired page to autom
  CAND WO-2001009766 (EN): Disclosed are methods for reducing the number of "pages", e.g

In [30]:

# What did the retriever actually return vs true relevant?
output_dir = results_df.iloc[0]['output_dir']
ret_df = pd.read_csv(f"{output_dir}/results.csv")

en_fail = recall0_topics[recall0_topics['language'] == 'EN']['topic'].tolist()[:2]

for topic_id in en_fail:
    top_ret = ret_df[ret_df['topic'] == topic_id].sort_values('score', ascending=False).head(3)
    true_cands = true_dict.get(topic_id, set())
    
    tf = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    tp = dataset.parse_patent([tf])[0]
    
    print(f"=== {topic_id} ===")
    print(f"  TOPIC abstract: {(tp.abstract_en or '')[:120]}")
    print(f"  TRUE relevant: {true_cands}")
    
    # Show top-3 retrieved (non-relevant) 
    print(f"  Top-3 RETRIEVED (non-relevant):")
    with sqlm.Session(engine) as session:
        for _, row in top_ret.iterrows():
            p = session.exec(sqlm.select(dataset.Patent).where(dataset.Patent.number == row['number'])).first()
            if p:
                lang = (p.language or 'en').lower()
                a = getattr(p, f"abstract_{lang}", None) or ''
                print(f"    {row['number']} (s={row['score']:.3f}): {a[:120]}")

    # Show scores of true relevant (if retrieved at all)
    print(f"  TRUE relevant scores:")
    for cid in true_cands:
        match = ret_df[(ret_df['topic'] == topic_id) & (ret_df['number'] == cid)]
        if len(match):
            rank = ret_df[ret_df['topic'] == topic_id].sort_values('score', ascending=False).reset_index()
            rank['rank'] = range(1, len(rank)+1)
            r = rank[rank['number'] == cid]['rank'].values[0]
            print(f"    {cid}: score={match['score'].values[0]:.4f}, rank={r}")
        else:
            print(f"    {cid}: NOT RETRIEVED AT ALL")
    print()


=== EP-1251440-A2 ===
  TOPIC abstract: A token type content providing system is provided, in which the system includes a portable user terminal and a linkup se
  TRUE relevant: {'WO-1998049813', 'WO-1998008056', 'EP-0987868', 'WO-1999066441'}
  Top-3 RETRIEVED (non-relevant):
    WO-2001093120 (s=0.812): A token process including storing transaction data for purchase of a product, generating a token for redemption of the p
    EP-1104973 (s=0.808): The invention relates to methods and systems for allowing
users of a cellular telecommunication system to obtain
service
    WO-2002021794 (s=0.799): Methods and systems for providing requested information over a computer network are presented. The invention uses machin
  TRUE relevant scores:
    WO-1998049813: score=0.6733, rank=362
    WO-1998008056: NOT RETRIEVED AT ALL
    EP-0987868: NOT RETRIEVED AT ALL
    WO-1999066441: score=0.6546, rank=648

=== EP-1255207-A2 ===
  TOPIC abstract: A method and apparatus for automatically accessin

In [32]:

# Side-by-side: topic abstract vs true relevant candidate abstracts (EN only)
en_fail = recall0_topics[recall0_topics['language'] == 'EN']['topic'].tolist()[:3]

for topic_id in en_fail:
    tf = next(test_topics_path.parent.glob(f"PAC_topics/files/{topic_id}*.xml"))
    tp = dataset.parse_patent([tf])[0]
    true_cands = list(true_dict.get(topic_id, set()))

    print(f"{'='*70}")
    print(f"TOPIC {topic_id}: {(tp.abstract_en or '')[:180]}")
    print()

    with sqlm.Session(engine) as session:
        for cid in true_cands:
            p = session.exec(sqlm.select(dataset.Patent).where(dataset.Patent.number == cid)).first()
            if not p:
                print(f"  >> {cid}: NOT IN DB"); continue
            abs_en = p.abstract_en or ''
            lang = (p.language or 'en').lower()
            abs_native = getattr(p, f"abstract_{lang}", None) or ''
            # show EN abstract preferably; native if no EN
            show = abs_en if abs_en else abs_native
            print(f"  >> {cid} ({p.language}): {show[:180]}")
    print()


TOPIC EP-1251440-A2: A token type content providing system is provided, in which the system includes a portable user terminal and a linkup server, the portable user terminal includes: a part for obtain

  >> WO-1998049813 (EN): A destination website access system employs a bar code reader cooperating with a personal computer or workstation for accessing a unique destination website through a remote Intern
  >> WO-1998008056 (EN): A navigation aid for use with printed maps, the aid comprising: (a) a receiver of transmitted signals from a geographic position fixing system such as, for example, GPS, which rece
  >> EP-0987868 (EN): The present invention relates to navigation of the Internet by a two-way
interactive communication mobile device capable of wireless communication, via a
link server (300), with se
  >> WO-1999066441 (EN): An interactive data transfer system and method is provided. In embodiments of the invention, the data transfer system includes a computing device, and a data

In [ ]:

# Summary
print(f"""
=== Hard Topics Analysis ===

Scale of the problem:
- {len(recall0_topics)} out of {len(all_details)} topics have recall@100 = 0 ({len(recall0_topics)/len(all_details)*100:.0f}%)
- Language breakdown: EN={len(recall0_topics[recall0_topics['language']=='EN'])}, FR={len(recall0_topics[recall0_topics['language']=='FR'])}, DE={len(recall0_topics[recall0_topics['language']=='DE'])}
- These zero-recall topics heavily drag down the mean recall across all topics

Why retrieval fails — Example EP-1251440-A2:
- Query: A portable terminal scans a token (barcode) to connect to a linkup server and deliver digital content
- Top retrieved (score ~0.81): Patents about token-based purchase redemption, cellular services, ML-based info delivery
  → High textual similarity ("token", "content", "server") but WRONG inventions
- True relevant (score ~0.65, rank 362+): Barcode-based website access, GPS navigation for printed maps, mobile internet via link servers
  → Same functional concept (physical token → digital content) but COMPLETELY DIFFERENT vocabulary

Root cause:
- Dense retrieval ranks by surface-level semantic similarity (what the text says)
- Prior art relevance is based on shared technical mechanisms and functional overlap (what the invention does)
- Patent examiners find relevant prior art across different terminology; embedding models cannot
- This is NOT a language issue — English topics fail at similar rates
""")
